# Chapter 4 — Dictionaries

Source: *Effective Python*, 3rd Edition (Brett Slatkin), Chapter 4.

---


### Item 25 — Be Cautious when Relying on Dictionary Insertion Ordering

Since Python 3.7, dictionaries officially preserve insertion order (iterating gives keys in the order they were first added) — this is now a language guarantee, not an implementation detail. But relying on it is still risky when the dict came from somewhere that *doesn't* guarantee order (e.g., a `dict`-like object from another library, or JSON parsed with `object_pairs_hook`) — always check the actual source's contract rather than assuming.


In [1]:
d = {}
d['banana'] = 1
d['apple'] = 2
d['cherry'] = 3

print(list(d.keys()))    # guaranteed to be insertion order: banana, apple, cherry -- since 3.7

# Sorting explicitly if you need a DIFFERENT order than insertion:
print(sorted(d.keys()))

# A custom mapping-like class does NOT automatically inherit this guarantee --
# only real dict (and OrderedDict, which predates the guarantee) are covered
class CustomMapping:
    def __init__(self):
        self._data = {}
    def __setitem__(self, k, v):
        self._data[k] = v
    def keys(self):
        return self._data.keys()   # delegates to a real dict here, but a different backing store might not

cm = CustomMapping()
cm['x'] = 1
cm['y'] = 2
print(list(cm.keys()))


['banana', 'apple', 'cherry']
['apple', 'banana', 'cherry']
['x', 'y']


### Item 26 — Prefer `get` over `in` and `KeyError` to Handle Missing Dictionary Keys

Three ways to handle a possibly-missing dictionary key: check `if key in d` first (two lookups: the check, then the access), catch `KeyError` (works, but exceptions-as-control-flow reads oddly for something this common), or use `d.get(key, default)` (one lookup, one line, a default value baked right in). `.get()` is the book's preferred default for the common case.


In [2]:
votes = {'baguette': 4, 'ciabatta': 2}

# 'in' check: two dictionary lookups (membership test, then access)
bread = 'sourdough'
if bread in votes:
    count = votes[bread]
else:
    count = 0
print('via in:', count)

# KeyError: works, but reads oddly for something this routine
try:
    count = votes[bread]
except KeyError:
    count = 0
print('via except:', count)

# .get(): one lookup, one line, default built in -- the preferred default
count = votes.get(bread, 0)
print('via get:', count)


via in: 0
via except: 0
via get: 0


### Item 27 — Prefer `defaultdict` over `setdefault` to Handle Missing Items in Internal State

`d.setdefault(key, default)` returns the existing value or inserts+returns `default` if missing — but it always *constructs* the default value first, even when it isn't needed, which is wasteful for an expensive default (like a new list). `collections.defaultdict(factory)` instead calls `factory()` **lazily**, only when a key is actually missing, and reads more clearly at the call site since every access just looks like normal dict indexing.


In [3]:
from collections import defaultdict

# setdefault: works, but constructs [] every call regardless of whether it's needed
groups_a = {}
for name, group in [('Ada', 'red'), ('Alan', 'blue'), ('Grace', 'red')]:
    groups_a.setdefault(group, []).append(name)
print(groups_a)

# defaultdict: the list factory only runs when a key is actually missing
groups_b = defaultdict(list)
for name, group in [('Ada', 'red'), ('Alan', 'blue'), ('Grace', 'red')]:
    groups_b[group].append(name)   # reads like plain indexing -- no visible fallback logic
print(dict(groups_b))


{'red': ['Ada', 'Grace'], 'blue': ['Alan']}
{'red': ['Ada', 'Grace'], 'blue': ['Alan']}


### Item 28 — Know How to Construct Key-Dependent Default Values with `__missing__`

`defaultdict`'s factory function takes no arguments — it can't know *which* key was missing. When the default value genuinely depends on the key itself (e.g., "open a file whose name is the key"), define a custom dict subclass overriding `__missing__(self, key)`: Python calls it automatically (with the key) only when a lookup fails, and whatever it returns becomes the answer for that access.


In [4]:
class AutoLabelDict(dict):
    def __missing__(self, key):
        value = f'auto-generated-label-for-{key}'   # the default DEPENDS on which key was missing
        self[key] = value    # cache it, so next time it's a normal, fast lookup
        return value

labels = AutoLabelDict()
labels['explicit'] = 'Explicit Label'

print(labels['explicit'])          # normal lookup, no __missing__ involved
print(labels['auto_generated'])    # triggers __missing__, key-dependent default created
print(labels)                       # now cached in the dict for next time


Explicit Label
auto-generated-label-for-auto_generated
{'explicit': 'Explicit Label', 'auto_generated': 'auto-generated-label-for-auto_generated'}


### Item 29 — Compose Classes Instead of Deeply Nesting Dictionaries, Lists, and Tuples

A dict of dicts of lists of tuples can model almost anything, and that's exactly the problem — beyond two levels of nesting, the structure becomes unreadable and error-prone (what's at `data['a'][2][1]`? you have to trace it manually). Once nesting gets deep, switch to real classes: `@dataclass`-based objects with named attributes self-document what each piece of data means, and give you a natural place to add validation or methods.


In [5]:
from dataclasses import dataclass, field

# Deeply nested: technically works, but meaning is opaque without tracing it
gradebook_nested = {
    'Ada': {'Math': [90, 85], 'Physics': [88]},
}
print(gradebook_nested['Ada']['Math'][0])   # what IS this? you have to already know the shape

# Composed classes: the same data, but self-documenting
@dataclass
class Student:
    name: str
    grades: dict = field(default_factory=dict)   # {subject: [scores]}

    def average(self, subject):
        scores = self.grades.get(subject, [])
        return sum(scores) / len(scores) if scores else 0.0

ada = Student('Ada', {'Math': [90, 85], 'Physics': [88]})
print(ada.name, '->', ada.average('Math'))   # self-explanatory at the call site


90
Ada -> 87.5
